# Explore Taper Tools

Consolidated notebook for all taper-tool diagnostics:
1. NaN audit by suffix
2. Beam limits and FOV
3. Check problematic images
4. Beam propeBluries scatter (vs. redshift)
5. Tapering histograms
6. One-source blur diagnostic
7. Visual QC via data loader

In [13]:
# ============================================================
# CONFIG — edit here, everything below picks up from these
# ============================================================

# Root directory containing per-source subdirectories of raw FITS files
PSZ2_BASE = "/users/mbredber/scratch/data/PSZ2"
FITS_ROOT = PSZ2_BASE + "/fits"

# CSV with columns 'slug,z' (one row per source)
Z_CSV = PSZ2_BASE + "/cluster_source_data.csv"

# Processed-file mode configuration
CROP_MODE   = 'beam_crop'    # 'beam_crop' | 'fov_crop' | 'beam_crop_no_sub' | 'cheat_crop'
BLUR_METHOD = 'circular'     # 'circular'  | 'circular_no_sub' | 'cheat'
# Filename tag used in processed FITS (existing files use Blur; change to 'Blur' once regenerated)
BLUR_TAG    = 'Blur'           # 'Blur' for existing files, 'Blur' once new files are generated

from pathlib import Path as _P
PROCESSED_DIR = str(_P(PSZ2_BASE) / CROP_MODE / BLUR_METHOD / 'fits_files')
FILE_SUFFIX   = BLUR_METHOD   # suffix in FITS filenames, e.g. 'circular'

# How many source pairs to sample for beam-property / Blur diagnostics
N_PAIRS = 50          # set to 10**6 to use all available
RANDOM_SEED = 42

# Blur quicklook: FOV in arcmin (None = use native T50 FOV)
Blur_FOV_ARCMIN = 20.0

# Sections 4-5: which x/y/color axis to show in the beam-properties scatter
X_AXIS = "omega_tgt_raw"
Y_AXIS = "omega_tgt_T50"
C_AXIS = "cosmo_distance_Mpc"

# Section 5: which quantities to histogram
QUANT_FUNCS = {
    "ratio_tgt_circ_over_tgt_raw": lambda r: r["omega_tgt_circ"] / (r["omega_tgt_raw"] + 1e-30),
    "ratio_tgt_T50_over_tgt_raw":  lambda r: r["omega_tgt_T50"]  / (r["omega_tgt_raw"] + 1e-30),
}

# Section 7: data-loader config
GALAXY_CLASSES = [50, 51]
VERSIONS = [f"{BLUR_TAG}50kpc", "T50kpc"]
N_SHOW = 10

# Section 8: multi-scale blurring visual comparison
TARGET_KPC  = 50           # which kpc scale to highlight: 25, 50, or 100
SCALES      = [25, 50, 100]  # all scales to show in grid
N_BLURRING  = 3            # how many sources to show

In [14]:
# ============================================================
# SHARED IMPORTS
# ============================================================
import os, re, glob
from collections import defaultdict
from pathlib import Path

import numpy as np
from astropy.io import fits
from astropy.cosmology import Planck18 as COSMO
import astropy.units as u
from astropy.convolution import convolve_fft
from scipy.optimize import curve_fit

import matplotlib
matplotlib.use("Agg")          # non-interactive; switch to inline if in JupyterLab
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm
import matplotlib as mpl


try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

mpl.rcParams.update({
    "font.size": 12, "axes.titlesize": 12, "axes.labelsize": 12,
    "xtick.labelsize": 11, "ytick.labelsize": 11,
    "figure.titlesize": 13, "legend.fontsize": 11,
})

In [15]:
# ============================================================
# SHARED FITS / WCS UTILITIES
# ============================================================

from dcreclass.utils import (
    ARCSEC_PER_RAD, arcsec_per_pix, fwhm_major_as,
    beam_cov_world, beam_solid_angle_sr, kernel_from_beams,
    read_fits_array_header_wcs, reproject_like, robust_vmin_vmax,
)
from dcreclass.data import load_z_table, find_pairs_in_tree
from dcreclass.data.processing import circular_cov_kpc


def load_fits(path):
    """Read FITS → (2-D float array, header)."""
    arr, hdr, _ = read_fits_array_header_wcs(path)
    return arr, hdr


def pix_scales_rad(hdr):
    """Pixel scales in radians (used for FFT axes)."""
    asx, asy = arcsec_per_pix(hdr)
    return asx / ARCSEC_PER_RAD, asy / ARCSEC_PER_RAD


def center_crop_at(arr, ny_target, nx_target, cy, cx):
    ny, nx = arr.shape
    y0 = max(0, min(int(round(cy - ny_target/2)), ny - ny_target))
    x0 = max(0, min(int(round(cx - nx_target/2)), nx - nx_target))
    return arr[y0:y0+ny_target, x0:x0+nx_target]


def _wcs_center(hdr):
    try:
        from astropy.wcs import WCS
        w = WCS(hdr).celestial
        nx, ny = float(hdr['NAXIS1']), float(hdr['NAXIS2'])
        ra, dec = w.wcs_pix2world([[nx/2.0, ny/2.0]], 0)[0]
        return float(ra), float(dec)
    except Exception:
        return np.nan, np.nan


def pick_random_pairs(root_dir, n=5, seed=42, desired_kpc=50):
    """Find (RAW, T{desired_kpc}kpc) FITS pairs via find_pairs_in_tree."""
    import random
    pairs = [(raw, t) for _, raw, t, _, _ in find_pairs_in_tree(Path(root_dir), desired_kpc)]
    if not pairs:
        raise FileNotFoundError(f'No (RAW, T{desired_kpc}kpc) pairs found under {root_dir}')
    rng = random.Random(seed)
    return pairs if len(pairs) <= n else rng.sample(pairs, n)


print(f'Shared utilities loaded. ARCSEC_PER_RAD={ARCSEC_PER_RAD}')

Shared utilities loaded. ARCSEC_PER_RAD=206264.80624709636


## 1. NaN Audit by Suffix

Groups all FITS files under `PROCESSED_DIR` by their filename suffix pattern
and reports aggregate NaN statistics per group.

In [16]:
def _count_nans(fits_path):
    try:
        with fits.open(fits_path, memmap=False) as hdul:
            data = hdul[0].data
            if data is None: return None, None, None
            arr = np.squeeze(np.asarray(data, float))
            if arr.ndim == 3: arr = np.nanmean(arr, axis=0)
            if arr.ndim != 2: return None, None, None
            total = arr.size
            nan_count = int(np.isnan(arr).sum())
            return total, nan_count, nan_count/total*100
    except Exception as e:
        print(f'Error reading {fits_path}: {e}')
        return None, None, None


def _suffix_type(fits_path):
    name = fits_path.name
    for pat in ('_RAW_', f'_{BLUR_TAG}', '_T'):
        idx = name.find(pat)
        if idx != -1: return name[idx:]
    return name


proc_path = Path(PROCESSED_DIR)
fits_files = list(proc_path.glob('**/*.fits'))
print(f'Found {len(fits_files)} FITS files in {PROCESSED_DIR}')

suffix_stats = defaultdict(lambda: dict(count=0, files_with_nans=0,
                                        total_pixels=0, total_nans=0,
                                        max_nan_pct=0.0, worst_file=None))

for fp in tqdm(fits_files):
    suffix = _suffix_type(fp)
    total, nan_count, nan_pct = _count_nans(fp)
    if total is None: continue
    s = suffix_stats[suffix]
    s['count']        += 1
    s['total_pixels'] += total
    s['total_nans']   += nan_count
    if nan_count > 0:
        s['files_with_nans'] += 1
        if nan_pct > s['max_nan_pct']:
            s['max_nan_pct'] = nan_pct
            s['worst_file']  = fp.name

# --- Report ---
print('\n' + '='*80)
print('ALL FILE TYPES (sorted by file count)')
print('='*80)
for sfx, s in sorted(suffix_stats.items(), key=lambda x: x[1]['count'], reverse=True):
    avg = s['total_nans']/s['total_pixels']*100 if s['total_pixels'] else 0
    status = '  CLEAN' if s['total_nans']==0 else f'  {avg:.2f}% NaNs'
    print(f"  {sfx:55s} | {s['count']:4d} files | {status}")

print('\n' + '='*80)
print('PROBLEMATIC TYPES (avg NaN rate > 0.1%)')
print('='*80)
for sfx, s in sorted(suffix_stats.items(),
                      key=lambda x: x[1]['total_nans']/max(x[1]['total_pixels'],1),
                      reverse=True):
    rate = s['total_nans']/max(s['total_pixels'],1)
    if rate <= 0.001: continue
    aff  = s['files_with_nans']/max(s['count'],1)*100
    print(f"  {sfx}")
    print(f"    NaN rate: {rate*100:.2f}%  |  affected: {s['files_with_nans']}/{s['count']} ({aff:.1f}%)")
    if s['worst_file']: print(f"    worst: {s['worst_file']}")

Found 2760 FITS files in /users/mbredber/scratch/data/PSZ2/beam_crop/circular/fits_files


  0%|          | 0/2760 [00:00<?, ?it/s]

100%|██████████| 2760/2760 [00:01<00:00, 1650.91it/s]


ALL FILE TYPES (sorted by file count)
  _T25kpc_fmt_128x128_circular.fits                       |  276 files |   CLEAN
  _Blur50kpc_fmt_128x128_circular.fits                    |  276 files |   CLEAN
  _RAW_fmt_128x128_circular.fits                          |  276 files |   CLEAN
  _T100kpc_fmt_128x128_circular.fits                      |  276 files |   CLEAN
  _Blur100kpc_fmt_128x128_circular.fits                   |  276 files |   CLEAN
  _T50kpc_fmt_128x128_circular.fits                       |  276 files |   CLEAN
  _T100kpcSUB_fmt_128x128_circular.fits                   |  276 files |   CLEAN
  _Blur25kpc_fmt_128x128_circular.fits                    |  276 files |   CLEAN
  _T50kpcSUB_fmt_128x128_circular.fits                    |  276 files |   CLEAN
  _T25kpcSUB_fmt_128x128_circular.fits                    |  276 files |   CLEAN

PROBLEMATIC TYPES (avg NaN rate > 0.1%)


## 2. Beam Limits and FOV

For each `T{Y}kpc.fits` scale found under `FITS_ROOT`, computes the minimum
`n_beams = min(FOV_x, FOV_y) / FWHM` and minimum FOV side across all sources.
These minima set the "bottleneck" for equal-beam cropping.

In [17]:
PAT_SCALE = re.compile(r'T([0-9]+(?:\.[0-9]+)?)kpc\.fits$', re.IGNORECASE)

def _analyze_t_file(fpath):
    m = PAT_SCALE.search(fpath.name)
    if not m: return None
    scale = float(m.group(1))
    try:
        with fits.open(fpath, memmap=False) as hdul:
            hdr = hdul[0].header
            nax1, nax2 = int(hdr.get('NAXIS1',0)), int(hdr.get('NAXIS2',0))
            if nax1<=0 or nax2<=0: raise RuntimeError('Invalid NAXIS')
            fwhm_as = fwhm_major_as(hdr)
            if fwhm_as<=0: raise RuntimeError('Invalid BMAJ/BMIN')
            asx, asy = arcsec_per_pix(hdr)
            fov_min_as = min(nax1*asx, nax2*asy)
            n_beams = fov_min_as / max(fwhm_as, 1e-12)
            return scale, n_beams, fov_min_as/60.0, dict(path=str(fpath),
                nax1=nax1, nax2=nax2, asx=asx, asy=asy, fwhm_as=fwhm_as)
    except Exception as e:
        print(f'[WARN] {fpath}: {e}')
        return None


results_per_scale = {}
for f in sorted(Path(FITS_ROOT).rglob('*T*kpc.fits')):
    res = _analyze_t_file(f)
    if res is None: continue
    scale, n_beams, fov_arcmin, details = res
    results_per_scale.setdefault(scale, []).append((n_beams, fov_arcmin, details))

if not results_per_scale:
    print('No T*kpc.fits files found.')
else:
    print(f'Per-scale minima (over {sum(len(v) for v in results_per_scale.values())} files):\n')
    hdr = f'{"scale(kpc)":>10}  {"min n_beams":>12}  {"min FOV(arcmin)":>16}'
    print(hdr); print('-'*len(hdr))
    g_nbeams = (1e9, None, None)
    g_fov    = (1e9, None, None)
    for scale in sorted(results_per_scale):
        lst = results_per_scale[scale]
        min_n   = min(lst, key=lambda x: x[0])
        min_fov = min(lst, key=lambda x: x[1])
        print(f'  {scale:8.1f}  {min_n[0]:12.3f}  {min_fov[1]:16.3f}')
        if min_n[0]   < g_nbeams[0]: g_nbeams = (min_n[0],   scale, min_n[2])
        if min_fov[1] < g_fov[0]:    g_fov    = (min_fov[1], scale, min_fov[2])
    print(f'\nGlobal minimum n_beams = {g_nbeams[0]:.3f} at scale {g_nbeams[1]} kpc')
    print(f'  source: {g_nbeams[2]["path"]}')
    print(f'  FWHM={g_nbeams[2]["fwhm_as"]:.2f}"')
    print(f'\nGlobal minimum FOV = {g_fov[0]:.3f} arcmin at scale {g_fov[1]} kpc')
    print(f'  source: {g_fov[2]["path"]}')

Per-scale minima (over 828 files):

scale(kpc)   min n_beams   min FOV(arcmin)
------------------------------------------
      25.0        70.888            24.000
      50.0        55.090            17.933
     100.0        30.546            17.067

Global minimum n_beams = 30.546 at scale 100.0 kpc
  source: /users/mbredber/scratch/data/PSZ2/fits/PSZ2G048.10+57.16/PSZ2G048.10+57.16T100kpc.fits
  FWHM=117.89"

Global minimum FOV = 17.067 arcmin at scale 100.0 kpc
  source: /users/mbredber/scratch/data/PSZ2/fits/PSZ2G099.86+58.45/PSZ2G099.86+58.45T100kpc.fits


## 3. Check Problematic Images

Quick per-file diagnostic for specific suffix patterns: prints min, max, NaN
count, and zero count for the first `N_CHECK` files of each pattern.

In [18]:
N_CHECK = 10   # how many files to inspect per pattern

CHECK_PATTERNS = [
    '**/*_T25kpcSUB_fmt_128x128_circular.fits',
    '**/*_T100kpcSUB_fmt_128x128_circular.fits',
    '**/*_T100kpcSUB_fmt_128x128_circular_new.fits',
]

for pat in CHECK_PATTERNS:
    files = sorted(Path(PROCESSED_DIR).glob(pat))[:N_CHECK]
    print(f'\nPattern: {pat}  ({len(files)} files checked)')
    if not files:
        print('  (none found)')
        continue
    for f in files:
        with fits.open(f) as hdul:
            data = np.asarray(hdul[0].data, dtype=float)
        print(f'  {f.name}:')
        print(f'    min={np.nanmin(data):.4e}  max={np.nanmax(data):.4e}'
              f'  mean={np.nanmean(data):.4e}  std={np.nanstd(data):.4e}')
        print(f'    zeros={int((data==0).sum())}  '
              f'finite={int(np.isfinite(data).sum())}/{data.size}')


Pattern: **/*_T25kpcSUB_fmt_128x128_circular.fits  (10 files checked)
  PSZ2G023.17+86.71_T25kpcSUB_fmt_128x128_circular.fits:
    min=-6.4066e-04  max=1.1198e-03  mean=2.6175e-05  std=1.1962e-04
    zeros=0  finite=16384/16384
  PSZ2G031.93+78.71_T25kpcSUB_fmt_128x128_circular.fits:
    min=-9.3376e-04  max=7.5345e-03  mean=1.2228e-04  std=4.6814e-04
    zeros=0  finite=16384/16384
  PSZ2G033.81+77.18_T25kpcSUB_fmt_128x128_circular.fits:
    min=-2.5137e-03  max=6.5303e-03  mean=6.6832e-05  std=4.0067e-04
    zeros=0  finite=16384/16384
  PSZ2G040.58+77.12_T25kpcSUB_fmt_128x128_circular.fits:
    min=-3.1727e-03  max=1.5814e-03  mean=3.8750e-05  std=2.9230e-04
    zeros=0  finite=16384/16384
  PSZ2G045.13+67.78_T25kpcSUB_fmt_128x128_circular.fits:
    min=-9.4465e-04  max=7.0744e-04  mean=1.1863e-05  std=1.3191e-04
    zeros=0  finite=16384/16384
  PSZ2G045.87+57.70_T25kpcSUB_fmt_128x128_circular.fits:
    min=-4.1046e-04  max=7.3417e-04  mean=1.4112e-05  std=1.1187e-04
    zeros=0  

## 4. Beam Properties Scatter

For each (RAW, T50kpc) pair, compute `get_beam_info` (a rich set of beam and
kernel metrics). Then show a scatter of `Y_AXIS` vs `X_AXIS`, coloured by `C_AXIS`,
with marginal histograms and linear / exponential fits.

In [19]:
# ---- Beam-property helper functions ----

def _print_wcs_info(tag, hdr):
    dx, dy = pix_scales_rad(hdr)
    ra, dec = _wcs_center(hdr)
    print(f'[WCS] {tag}: {int(hdr["NAXIS1"])}x{int(hdr["NAXIS2"])} px | '
          f'{dx*206265:.4f}x{dy*206265:.4f} arcsec/px | '
          f'center RA={ra:.4f} Dec={dec:.4f}')


def get_beam_info(raw_fits, T50_fits, z, fwhm_kpc=50.0):
    """Compute ~30 beam/kernel metrics for one (RAW, T50) pair at redshift z."""
    I, Hraw = load_fits(raw_fits)
    _, Htgt  = load_fits(T50_fits)

    bmaj_RAW = float(Hraw['BMAJ']); bmin_RAW = float(Hraw['BMIN'])
    bmaj_T50 = float(Htgt['BMAJ']); bmin_T50 = float(Htgt['BMIN'])
    q_T50 = min(bmaj_T50,bmin_T50)/max(bmaj_T50,bmin_T50) if bmaj_T50>0 and bmin_T50>0 else np.nan
    e_T50 = 1.0 - q_T50
    pa_raw = float(Hraw.get('BPA', np.nan))
    pa_tgt = float(Htgt.get('BPA', np.nan))
    def _dpa(a,b):
        if not (np.isfinite(a) and np.isfinite(b)): return np.nan
        return abs((a-b+90.0)%180.0-90.0)

    C_raw_w   = beam_cov_world(Hraw)
    C_tgt_hdr = beam_cov_world(Htgt)
    Cker_circ = circular_cov_kpc(z, fwhm_kpc=fwhm_kpc)
    Cker_cheat = C_tgt_hdr - C_raw_w

    omega = lambda C: 2.0*np.pi*np.sqrt(max(0.0, np.linalg.det(C))) if C is not None else np.nan
    omega_raw   = beam_solid_angle_sr(Hraw)
    omega_tgt   = beam_solid_angle_sr(Htgt)
    omega_circ  = omega(Cker_circ)
    omega_cheat = omega(Cker_cheat)
    omega_tgt_circ = omega(C_raw_w + Cker_circ) if Cker_circ is not None else np.nan

    DA_kpc = COSMO.angular_diameter_distance(z).to_value(u.kpc) if (z is not None and np.isfinite(z) and z>0) else np.nan
    scale_phys = DA_kpc**2 if np.isfinite(DA_kpc) else np.nan

    C_pred = C_raw_w + Cker_circ if Cker_circ is not None else C_raw_w
    Delta  = C_tgt_hdr - C_pred
    cov_mismatch_frob  = float(np.linalg.norm(Delta,'fro')/(np.linalg.norm(C_tgt_hdr,'fro')+1e-30))
    wt, Blur = np.linalg.eigh(C_tgt_hdr)
    Delta_t = Blur.T @ Delta @ Blur
    ellip_mis = float((Delta_t[0,0]-Delta_t[1,1])/((wt[0]+wt[1])+1e-30))
    pa_mis    = float(Delta_t[0,1]/(0.5*(wt[0]+wt[1])+1e-30))

    taper_leverage = float(np.linalg.norm(Cker_circ,'fro')/(np.linalg.norm(C_raw_w,'fro')+1e-30)) if Cker_circ is not None else np.nan

    # uv-kernel FWHM from cheat kernel
    w_cheat, _ = np.linalg.eigh(np.maximum(Cker_cheat, 0))
    sig_cheat   = np.sqrt(np.maximum(w_cheat, 0.0))
    sig_uv      = 1.0/(2.0*np.pi*np.maximum(sig_cheat,1e-30))
    fwhm_uv_kl  = np.sort((2.0*np.sqrt(2.0*np.log(2.0)))*sig_uv/1e3)

    return {
        'z': float(z),
        'cosmo_distance_Mpc': DA_kpc/1e3,
        'omega_tgt_raw': float(omega_raw),
        'omega_tgt_T50': float(omega_tgt),
        'omega_tgt_circ': float(omega_tgt_circ),
        'omega_rt_circ':  float(omega_circ),
        'omega_rt_cheat': float(omega_cheat),
        'omega_tgt_raw_phys_kpc2': float(omega_raw*scale_phys),
        'ratio_tgt_circ_over_tgt_raw': float(omega_tgt_circ/(omega_raw+1e-30)),
        'ratio_tgt_T50_over_tgt_raw':  float(omega_tgt/(omega_raw+1e-30)),
        'ratio_rt_circ_over_rt_cheat': float(omega_circ/(omega_cheat+1e-30)),
        'e_tgt': float(e_T50), 'q_tgt': float(q_T50),
        'pa_raw': float(pa_raw), 'pa_tgt': float(pa_tgt),
        'dpa_deg': float(_dpa(pa_tgt, pa_raw)),
        'beam_major_T50': bmaj_T50, 'beam_minor_T50': bmin_T50,
        'beam_major_RAW': bmaj_RAW, 'beam_minor_RAW': bmin_RAW,
        'beam_major_ratio': float(bmaj_T50/(bmaj_RAW+1e-30)),
        'beam_minor_ratio': float(bmin_T50/(bmin_RAW+1e-30)),
        'uvker_fwhm_minor_kl': float(fwhm_uv_kl[0]),
        'uvker_fwhm_major_kl': float(fwhm_uv_kl[1]),
        'uvker_fwhm_geo_kl':   float(np.sqrt(fwhm_uv_kl[0]*fwhm_uv_kl[1])),
        'uvker_axis_ratio': float(fwhm_uv_kl[0]/(fwhm_uv_kl[1]+1e-30)),
        'cov_mismatch_frob': cov_mismatch_frob,
        'ellip_mis': ellip_mis, 'pa_mis': pa_mis,
        'taper_leverage': taper_leverage,
    }

In [ ]:
# ---- Run beam-info collection ----
slug_to_z = load_z_table(Z_CSV)

# find_pairs_in_tree returns ALL (slug, RAW, T50, ...) tuples under FITS_ROOT;
# no random subsampling — N_PAIRS / RANDOM_SEED are ignored here.
all_pair_tuples = find_pairs_in_tree(Path(FITS_ROOT), desired_kpc=50)
pairs = [(raw, t50) for _slug, raw, t50, *_ in all_pair_tuples]
print(f'Processing {len(pairs)} pairs (full dataset)...')

beam_results = []
for RAW, T50 in tqdm(pairs):
    slug = Path(RAW).stem
    z = slug_to_z.get(slug, np.nan)
    if not np.isfinite(z): continue
    try:
        beam_results.append(get_beam_info(RAW, T50, z=z, fwhm_kpc=50.0))
    except Exception as e:
        print(f'[error] {slug}: {e}')

print(f'Collected {len(beam_results)} results.')

# ---- Label map for axis labels ----
LABEL_MAP = {
    'z':                          r'Redshift $z$',
    'cosmo_distance_Mpc':         r'Angular diameter distance $D_\mathrm{A}$ [Mpc]',
    'omega_tgt_circ':             r'Target-circ beam area $\Omega$ [sr]',
    'omega_tgt_raw':              r'Reference beam area $\Omega_\mathrm{ref}$ [$10^{-10}$ sr]',
    'omega_tgt_T50':              r'Tap. $50\,$kpc target beam area $\Omega_\mathrm{T50}$ [sr]',
    'omega_rt_circ':              r'Blur $50\,kpc beam area $\Omega_\mathrm{rt}$ [sr]',
    'ratio_tgt_circ_over_tgt_raw':r'$\Omega_\mathrm{tgt\text{-}circ}\ /\ \Omega_\mathrm{raw}$',
    'ratio_tgt_T50_over_tgt_raw': r'$\Omega_\mathrm{T50}\ /\ \Omega_\mathrm{raw}$',
    'ratio_rt_circ_over_rt_cheat':r'$\Omega_\mathrm{rt\text{-}circ}\ /\ \Omega_\mathrm{rt\text{-}cheat}$',
    'e_tgt':                      r'Ellipticity $e = 1 - b/a$',
    'q_tgt':                      r'Axis ratio $q = b/a$',
    'dpa_deg':                    r'PA misalignment $|\Delta\mathrm{PA}|$ [deg]',
    'beam_major_ratio':           r'Beam major-axis ratio $a_\mathrm{T50}/a_\mathrm{raw}$',
    'beam_minor_ratio':           r'Beam minor-axis ratio $b_\mathrm{T50}/b_\mathrm{raw}$',
    'uvker_fwhm_geo_kl':          r'UV-kernel geometric FWHM [k$\lambda$]',
    'uvker_axis_ratio':           r'UV-kernel axis ratio $b/a$',
    'cov_mismatch_frob':          r'Cov. mismatch $\|\Delta C\|_F / \|C_\mathrm{tgt}\|_F$',
    'ellip_mis':                  r'Ellipticity mismatch $\Delta e$',
    'pa_mis':                     r'PA mismatch (target frame)',
    'taper_leverage':             r'Taper leverage $\|C_{50}\|_F / \|C_\mathrm{raw}\|_F$',
}

# ---- Static scatter with marginals ----
if beam_results:
    if X_AXIS == 'omega_tgt_raw':
        X = np.array([r[X_AXIS] for r in beam_results], float) * 1e10  # convert to units of 1e-10 sr
    else:
        X = np.array([r[X_AXIS] for r in beam_results], float)
    Y = np.array([r[Y_AXIS] for r in beam_results], float)
    C = np.array([r[C_AXIS] for r in beam_results], float)

    mask = np.isfinite(X) & np.isfinite(Y)
    Xf, Yf, Cf = X[mask], Y[mask], C[mask]

    fig = plt.figure(figsize=(7.0, 5.5), layout='constrained')
    gs  = fig.add_gridspec(2, 3, width_ratios=[4,1,0.15], height_ratios=[1,4],
                            wspace=0.05, hspace=0.05)
    ax       = fig.add_subplot(gs[1, 0])
    ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
    ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)
    cax      = fig.add_subplot(gs[1, 2])

    Cf_finite = Cf[np.isfinite(Cf)]
    vmin = float(np.nanpercentile(Cf_finite, 2))  if Cf_finite.size else 0.0
    vmax = float(np.nanpercentile(Cf_finite, 98)) if Cf_finite.size else 1.0

    # Build second Y series (RT/circ) aligned to the same X points
    Y2 = np.array([r['omega_rt_circ'] for r in beam_results], float)
    Xf2, Y2f, Cf2 = X[mask], Y2[mask], C[mask]

    # Draw vertical connectors between T50 (circle) and RT-circ (triangle) per source
    for xi, y1i, y2i in zip(Xf, Yf, Y2f):
        if np.isfinite(y1i) and np.isfinite(y2i):
            ax.plot([xi, xi], [y1i, y2i], color='0.6', lw=0.6, zorder=1)

    # T50kpc — circles (primary Y_AXIS series)
    sc = ax.scatter(Xf, Yf, c=Cf, s=24, alpha=0.85, cmap='viridis',
                    vmin=vmin, vmax=vmax, marker='o', zorder=3,
                    label='Tap. $50\,kpc target beam')
    # Blur50kpc — triangles (omega_rt_circ)
    sc2 = ax.scatter(Xf2, Y2f, c=Cf2, s=24, alpha=0.85, cmap='viridis',
                    vmin=vmin, vmax=vmax, marker='^', zorder=3,
                    label='Blur $50\,kpc target beam')

    cb = fig.colorbar(sc, cax=cax)
    cb.set_label(LABEL_MAP.get(C_AXIS, C_AXIS).replace('D_A', r'$D_A$'))    
    ax.set_yscale('log')   # logarithmic y-axis
    
    cmap_v = plt.get_cmap('viridis')
    outline_col = cmap_v(0.55)
    from matplotlib.legend_handler import HandlerTuple

    # Combine marker + histogram line into one legend entry per series
    tap_marker  = plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='0.5', markersize=7)
    tap_line    = plt.Line2D([0],[0], linestyle='-',  color=outline_col, lw=1.2)
    blur_marker = plt.Line2D([0],[0], marker='^', color='w', markerfacecolor='0.5', markersize=7)
    blur_line   = plt.Line2D([0],[0], linestyle='--', color=outline_col, lw=1.2)

    ax.legend(
        handles=[(tap_marker,  tap_line),
                (blur_marker, blur_line)],
        labels=[r'Tap. $50\,$kpc $\Omega_\mathrm{T50}$',
                r'Blur $50\,$kpc $\Omega_\mathrm{blur}$'],
        handler_map={tuple: HandlerTuple(ndivide=None, pad=0.5)},
        frameon=False, fontsize=9,
    )
    ax.set_xlabel(LABEL_MAP.get(X_AXIS, X_AXIS))
    ax.set_ylabel(LABEL_MAP.get(Y_AXIS, Y_AXIS))
    ax.grid(True, which='both', lw=0.4, color='0.8', zorder=0)
    ax_histx.grid(False)
    ax_histy.grid(False)

    nbx = max(10, int(np.sqrt(Xf.size)))
    nby = max(10, int(np.sqrt(Yf.size)))
    cf_norm = (np.nanmedian(Cf) - vmin) / (vmax - vmin + 1e-30)
    hist_col = cmap_v(float(np.clip(cf_norm, 0, 1)))
    linx_bins = np.linspace(Xf.min(), Xf.max(), nbx)
    counts_x, edges_x = np.histogram(Xf, bins=linx_bins)
    for i, count in enumerate(counts_x):
        if count == 0: continue
        in_bin = (Xf >= edges_x[i]) & (Xf < edges_x[i+1])
        mean_c = np.mean(Cf[in_bin]) if in_bin.any() else vmin
        bar_col = cmap_v(float(np.clip((mean_c - vmin) / (vmax - vmin + 1e-30), 0, 1)))
        ax_histx.bar(edges_x[i], count, width=edges_x[i+1]-edges_x[i],
                    align='edge', color=bar_col, alpha=0.8, linewidth=0)
    
    logy_bins = np.logspace(np.log10(np.nanmin(Yf[Yf>0])), np.log10(np.nanmax(Yf)), nby)

    def _stepped_outline_y(ax, values, bins, linestyle, color):
        """Draw a horizontal histogram as a single stepped outline."""
        counts, edges = np.histogram(values, bins=bins)
        # Build step coordinates: start from bottom-left of first bin
        xs, ys = [0], [edges[0]]
        for c, e0, e1 in zip(counts, edges[:-1], edges[1:]):
            ys += [e0, e1]
            xs += [c,  c]
        xs += [0]
        ys += [edges[-1]]
        ax.plot(xs, ys, linestyle=linestyle, color=color, lw=1.2)

    # Use a neutral mid-viridis colour for both outlines
    _stepped_outline_y(ax_histy, Yf,             logy_bins, '-',  outline_col)
    _stepped_outline_y(ax_histy, Y2f[Y2f > 0],  logy_bins, '--', outline_col)

    plt.setp(ax_histx.get_xticklabels(), visible=False)
    plt.setp(ax_histy.get_yticklabels(), visible=False)
    
    ax_histx.set_yticks([])
    ax_histy.set_xticks([])
    ax_histy.xaxis.offsetText.set_visible(False)
    ax_histx.tick_params(axis='x', length=0)
    ax_histy.tick_params(axis='y', length=0)

    for spine in ax_histx.spines.values(): spine.set_visible(False)
    for spine in ax_histy.spines.values(): spine.set_visible(False)

    #fig.suptitle(f'{Y_AXIS} vs {X_AXIS}  (N={Xf.size})')
    plt.savefig(f'explore_blurring_outputs/{X_AXIS}_vs_{Y_AXIS}_vs_{C_AXIS}_scatter.pdf', dpi=300)
    print(f'Saved scatter plot to explore_blurring_outputs/{X_AXIS}_vs_{Y_AXIS}_vs_{C_AXIS}_scatter.pdf')

Processing 276 pairs (full dataset)...


100%|██████████| 276/276 [00:04<00:00, 62.87it/s]


Collected 276 results.
Saved scatter plot to explore_blurring_outputs/omega_tgt_raw_vs_omega_tgt_T50_vs_cosmo_distance_Mpc_scatter.pdf


## 5. Tapering Histograms

Overlaid, area-normalised histograms for each quantity in `QUANT_FUNCS`.
Uses the `beam_results` collected in Section 4 (re-run that cell first if needed).

In [ ]:
def _common_log_edges(vals_list, bins=30, clip=(0.5, 99.5)):
    pooled = np.concatenate([v for v in vals_list if v.size])
    pooled = pooled[np.isfinite(pooled) & (pooled > 0)]
    if pooled.size == 0: return None
    lo, hi = np.nanpercentile(pooled, list(clip))
    if not (np.isfinite(lo) and np.isfinite(hi) and 0 < lo < hi):
        lo = max(np.nanmin(pooled), np.nextafter(0.0, 1.0))
        hi = np.nanmax(pooled)
    return np.logspace(np.log10(lo), np.log10(hi), bins + 1)


if not beam_results:
    print('Run Section 4 first to populate beam_results.')
else:
    labels  = list(QUANT_FUNCS.keys())
    colors  = plt.rcParams['axes.prop_cycle'].by_key()['color']
    BINS    = 30

    # gather values
    vals_by_label = []
    for q in labels:
        v = np.array([QUANT_FUNCS[q](r) for r in beam_results], float)
        v = v[np.isfinite(v) & (v > 0)]
        vals_by_label.append(v)

    edges = _common_log_edges(vals_by_label, bins=BINS)
    if edges is None:
        print('No finite positive values found.')
    else:
        x = 0.5*(edges[:-1]+edges[1:])

        fig, ax = plt.subplots(figsize=(7.2, 4.4))
        for i, (q, v) in enumerate(zip(labels, vals_by_label)):
            if v.size < 2: continue
            c = colors[i % len(colors)]
            h, _ = np.histogram(v, bins=edges, density=True)
            y = h / (np.trapz(h, x) + 1e-30)    # area-normalise to 1
            med  = np.median(v)
            lo68 = np.percentile(v, 16)
            hi68 = np.percentile(v, 84)
            label_str = (LABEL_MAP.get(q, q) +
                         f'  med={med:.3g} [{lo68:.3g}, {hi68:.3g}]  N={v.size}')
            ax.fill_between(x, 0, y, color=c, alpha=0.22, step='mid')
            ax.plot(x, y, color=c, lw=1.8, drawstyle='steps-mid', label=label_str)

        ax.set_xscale('log')
        ax.set_xlabel('Quantity value')
        ax.set_ylabel('Normalised density')
        ax.set_title(f'Tapering histograms (N={len(beam_results)} sources)')
        ax.legend(handles=[*ax.get_legend_handles_labels()[0], *hist_handles],
          frameon=False, fontsize=9)
        fig.tight_layout()
        plt.savefig('explore_blurring_outputs/tapering_histograms.pdf', dpi=300)
        print('Saved tapering_histograms.pdf')

/tmp/ipykernel_27143/1191031737.py:37: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  y = h / (np.trapz(h, x) + 1e-30)    # area-normalise to 1


Saved tapering_histograms.pdf


## 6. One-Source RT Diagnostic

Full image-plane / uv-plane comparison for a single (RAW, T50kpc) pair:
- Row 1: I (RAW), I\*G (image-space), F⁻¹{F(I)·W} (uv-space), T50kpc, kernel G
- Row 2: log₁₀ amplitudes in uv plane
- Row 3: image-space residuals (gain-matched)
- Row 4: uv-amplitude residuals

Also saves separate UV-profile and phase-statistics figures.

In [ ]:
# ---- RT-specific utilities ----

def fft2c(A, dx, dy):
    A0 = np.nan_to_num(np.asarray(A, float), nan=0.0)
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(A0))) * (dx*dy)


def ifft2c(F, dx, dy):
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(F))).real / (dx*dy)


def uv_grid(ny, nx, dx, dy):
    u = np.fft.fftshift(np.fft.fftfreq(nx, d=dx))
    v = np.fft.fftshift(np.fft.fftfreq(ny, d=dy))
    return np.meshgrid(u, v)


def W_from_Cker(U, V, C_world):
    a,b,c = float(C_world[0,0]), float(C_world[0,1]), float(C_world[1,1])
    return np.exp(-2.0*(np.pi**2)*(a*(U*U)+2.0*b*(U*V)+c*(V*V)))


def robust_gain_match(ref, src, nsig=3.0, roi=0.85):
    ny, nx = ref.shape
    y0,y1 = int(ny*(1-roi)/2), int(ny*(1+roi)/2)
    x0,x1 = int(nx*(1-roi)/2), int(nx*(1+roi)/2)
    r,s = ref[y0:y1,x0:x1], src[y0:y1,x0:x1]
    m = np.isfinite(r) & np.isfinite(s)
    if not np.any(m): return 1.0, {'Npix':0,'rms':np.nan}
    for _ in range(3):
        rf,sf = r[m], s[m]
        g = float(np.dot(rf,sf)/(np.dot(sf,sf)+1e-30))
        res = rf - g*sf
        sig = float(np.std(res))
        m2  = np.abs(res) < nsig*sig
        if not np.any(m2): break
        m[m] = m2
    rf,sf = r[m],s[m]
    g = float(np.dot(rf,sf)/(np.dot(sf,sf)+1e-30))
    rms = float(np.sqrt(np.mean((rf-g*sf)**2)))
    return g, {'Npix': int(m.sum()), 'rms': rms}


def _radial_profiles(U, V, A_T, A_IG, A_IW, F_T, F_IG, nbins=28):
    R = np.hypot(U, V)
    r_max = float(np.nanmax(R))
    edges = np.linspace(0, r_max, nbins+1)
    r_cen = 0.5*(edges[:-1]+edges[1:])
    med_T, med_RT, med_IG = [], [], []
    mean_abs_dphi, sigma_phi = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (R >= lo) & (R < hi)
        if not np.any(m):
            med_T.append(np.nan); med_RT.append(np.nan); med_IG.append(np.nan)
            mean_abs_dphi.append(np.nan); sigma_phi.append(np.nan)
            continue
        med_T.append(float(np.nanmedian(A_T[m])))
        med_RT.append(float(np.nanmedian(A_IW[m])))
        med_IG.append(float(np.nanmedian(A_IG[m])))
        dphi = np.angle(F_T[m] * np.conj(F_IG[m]))
        mean_abs_dphi.append(float(np.nanmean(np.abs(dphi))))
        sigma_phi.append(float(np.nanstd(dphi)))
    kla_scale = 1e-3  # cycles/rad → kλ is not exact but consistent
    return dict(r_kla=np.array(r_cen)*kla_scale,
                med_T=np.array(med_T), med_RT=np.array(med_RT), med_IG=np.array(med_IG),
                mean_abs_dphi=np.array(mean_abs_dphi), sigma_phi=np.array(sigma_phi))


def make_blur_quicklook(raw_fits, t50_fits=None, out_stem='blur_quicklook',
                       fov_arcmin=None, per_image_stretch=False,
                       percentile_lo=1.0, percentile_hi=99.5):
    """Produce four figures for one (RAW, T50) pair and return radial-profile dict."""

    def _stretch01(x):
        a = np.asarray(x, float)
        lo, hi = np.nanpercentile(a, percentile_lo), np.nanpercentile(a, percentile_hi)
        return np.clip((a-lo)/(hi-lo+1e-12), 0, 1)

    I, Hraw = load_fits(raw_fits)
    dx, dy  = pix_scales_rad(Hraw)
    asx, asy = dx*ARCSEC_PER_RAD, dy*ARCSEC_PER_RAD

    # find T50
    if not t50_fits:
        base = Path(raw_fits).stem
        cands = sorted(glob.glob(os.path.join(os.path.dirname(raw_fits),
                                              f'{base}T50kpc*.fits')))
        assert cands, 'No T50kpc FITS found.'
        t50_fits = cands[0]
    T_native, Htgt = load_fits(t50_fits)
    T = reproject_like(T_native, Htgt, Hraw)

    # image-space convolution
    ker = kernel_from_beams(Hraw, Htgt)
    G_small = np.asarray(ker.array, float); G_small /= (G_small.sum()+1e-12)
    IG_full = convolve_fft(I, ker, boundary='fill', fill_value=np.nan,
                           nan_treatment='interpolate', normalize_kernel=True,
                           psf_pad=True, fft_pad=True, allow_huge=True)
    scale = beam_solid_angle_sr(Htgt)/beam_solid_angle_sr(Hraw)
    IG_full *= scale

    # uv-space route on full frame
    F_I_full = fft2c(I, dx, dy)
    Cker_w   = beam_cov_world(Htgt) - beam_cov_world(Hraw)
    U_full, V_full = uv_grid(*I.shape, dx, dy)
    Wuv_full = W_from_Cker(U_full, V_full, Cker_w)
    IU_full  = ifft2c(F_I_full*Wuv_full, dx, dy)*scale

    # crop
    if fov_arcmin is not None:
        nx_crop = min(int(round(fov_arcmin*60.0/asx)), I.shape[1])
        ny_crop = min(int(round(fov_arcmin*60.0/asy)), I.shape[0])
        m = min(nx_crop, ny_crop)
        nx_crop, ny_crop = m, m
    else:
        dxT, dyT = pix_scales_rad(Htgt)
        asxT, asyT = dxT*ARCSEC_PER_RAD, dyT*ARCSEC_PER_RAD
        nyT, nxT = T_native.shape
        nx_crop = min(int(round(nxT*asxT/asx)), I.shape[1])
        ny_crop = min(int(round(nyT*asyT/asy)), I.shape[0])

    cy, cx = (I.shape[0]-1)/2.0, (I.shape[1]-1)/2.0
    crop = lambda a: center_crop_at(a, ny_crop, nx_crop, cy, cx)
    I_c   = crop(I);     IG_c  = crop(IG_full);  IU_c = crop(IU_full)
    T_c   = crop(T)
    G_cvs = np.zeros_like(I, float)
    gy, gx = G_small.shape
    yc, xc = int(cy), int(cx)
    G_cvs[yc-gy//2:yc-gy//2+gy, xc-gx//2:xc-gx//2+gx] = G_small
    G_c   = crop(G_cvs)

    # uv on cropped
    F_I   = fft2c(I_c, dx, dy);  F_IG  = fft2c(IG_c, dx, dy)
    F_T   = fft2c(T_c, dx, dy)
    U, V  = uv_grid(ny_crop, nx_crop, dx, dy)
    Wuv   = W_from_Cker(U, V, Cker_w)
    FIW   = F_I*Wuv
    A_I   = np.abs(F_I);  A_IG = np.abs(F_IG)
    A_IW  = np.abs(FIW);  A_T  = np.abs(F_T)

    prof = _radial_profiles(U, V, A_T, A_IG, A_IW, F_T, F_IG)

    # gain match + residuals
    gIG, _ = robust_gain_match(T_c, IG_c); gIU, _ = robust_gain_match(T_c, IU_c)
    IGm, IUm = gIG*IG_c, gIU*IU_c
    R12, R13, R23 = IGm-IUm, IGm-T_c, IUm-T_c

    def _plot_row1_2(stem):
        imgs = [I_c*(beam_solid_angle_sr(Hraw)/beam_solid_angle_sr(Htgt)), IG_c, IU_c, T_c, G_c]
        titles = ['I (RAW)', r'$I*G$', r'$\mathcal{F}^{-1}\{\mathcal{F}(I)\,W\}$',
                  r'$T_{50\mathrm{kpc}}$', r'Kernel $G$']
        all_px = np.concatenate([x[np.isfinite(x)].ravel() for x in imgs])
        norm1  = mcolors.Normalize(vmin=float(np.nanpercentile(all_px,1)),
                                   vmax=float(np.nanpercentile(all_px,99.5)))
        log_uvs = [np.log10(a+1e-12) for a in [A_I, A_IG, A_IW, A_T]]
        uv_px  = np.concatenate([x[np.isfinite(x)].ravel() for x in log_uvs])
        norm2  = mcolors.Normalize(vmin=float(np.nanpercentile(uv_px,1)),
                                   vmax=float(np.nanpercentile(uv_px,99.5)))
        p99W = float(np.nanpercentile(np.log10(Wuv+1e-12), 99.5))
        sW   = 10.0**(norm2.vmax - p99W)
        log_uvs_all = log_uvs + [np.log10(sW*Wuv+1e-12)]
        titles2 = [r'$\log|\mathcal{F}(I)|$', r'$\log|\mathcal{F}(I*G)|$',
                   r'$\log|\mathcal{F}(I)\,W|$', r'$\log|\mathcal{F}(T)|$',
                   r'$\log\tilde{W}$ (scaled)']

        fig, axes = plt.subplots(2, 5, figsize=(13, 5.2))
        for j, (img, ttl) in enumerate(zip(imgs, titles)):
            axes[0,j].imshow(img if not per_image_stretch else _stretch01(img),
                             origin='lower', cmap='viridis', norm=norm1)
            axes[0,j].set_axis_off(); axes[0,j].set_title(ttl, fontsize=9)
        for j, (arr, ttl) in enumerate(zip(log_uvs_all, titles2)):
            axes[1,j].imshow(arr, origin='lower', cmap='viridis', norm=norm2)
            axes[1,j].set_axis_off(); axes[1,j].set_title(ttl, fontsize=9)
        fig.colorbar(ScalarMappable(norm=norm1, cmap='viridis'),
                     ax=axes[0,:], fraction=0.02, label='Jy/beam$_{tgt}$')
        fig.colorbar(ScalarMappable(norm=norm2, cmap='viridis'),
                     ax=axes[1,:], fraction=0.02, label=r'$\log_{10}|\mathcal{F}|$')
        fig.tight_layout()
        plt.savefig(f'{stem}_rows12.pdf', bbox_inches='tight', dpi=200)

    def _plot_row3_4(stem):
        rmax  = float(np.nanpercentile(np.concatenate([np.abs(r)[np.isfinite(r)].ravel()
                                                       for r in (R12,R13,R23)]), 99.0))
        norm3 = TwoSlopeNorm(vmin=-rmax, vcenter=0.0, vmax=rmax)
        D12,D13,D23 = np.abs(A_IG-A_IW), np.abs(A_IG-A_T), np.abs(A_IW-A_T)
        logDs = [np.log10(x+1e-12) for x in (D12,D13,D23)]
        all4  = np.concatenate([x[np.isfinite(x)].ravel() for x in logDs])
        norm4 = mcolors.Normalize(vmin=float(np.nanpercentile(all4,1)),
                                  vmax=float(np.nanpercentile(all4,99.5)))
        t3 = [r'$(I*G)-\mathcal{F}^{-1}\{\mathcal{F}(I)W\}$',
              r'$(I*G)-T_{50}$', r'$\mathcal{F}^{-1}\{\mathcal{F}(I)W\}-T_{50}$']
        t4 = [r'$\log|\,|\mathcal{F}(I*G)|-|\mathcal{F}(I)W|\,|$',
              r'$\log|\,|\mathcal{F}(I*G)|-|\mathcal{F}(T)|\,|$',
              r'$\log|\,|\mathcal{F}(I)W|-|\mathcal{F}(T)|\,|$']
        fig, axes = plt.subplots(2, 3, figsize=(11, 7.8))
        for j, (R, ttl) in enumerate(zip((R12,R13,R23), t3)):
            axes[0,j].imshow(R, origin='lower', cmap='coolwarm', norm=norm3)
            axes[0,j].set_axis_off(); axes[0,j].set_title(ttl, fontsize=10)
        for j, (arr, ttl) in enumerate(zip(logDs, t4)):
            axes[1,j].imshow(arr, origin='lower', cmap='viridis', norm=norm4)
            axes[1,j].set_axis_off(); axes[1,j].set_title(ttl, fontsize=10)
        fig.colorbar(ScalarMappable(norm=norm3, cmap='coolwarm'),
                     ax=axes[0,:], fraction=0.02, label='residual [Jy/beam]')
        fig.colorbar(ScalarMappable(norm=norm4, cmap='viridis'),
                     ax=axes[1,:], fraction=0.02, label=r'$\log_{10}|\Delta|$')
        fig.tight_layout()
        plt.savefig(f'{stem}_rows34.pdf', bbox_inches='tight', dpi=200)

    def _plot_uv_profiles(stem):
        r = prof['r_kla']
        fig, ax = plt.subplots(figsize=(5.8, 5.0))
        ax.plot(r, prof['med_T'],  ls='--', lw=1.8, color='C0', label=r'$|\mathcal{F}(T)|$')
        ax.plot(r, prof['med_RT'], ls='-',  lw=1.4, color='C1', label=r'$|\mathcal{F}(I)W|$')
        ax.plot(r, prof['med_IG'], ls='-',  lw=1.4, color='C2', label=r'$|\mathcal{F}(I*G)|$')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel(r'baseline $r$ (kλ)'); ax.set_ylabel('amplitude (Jy)')
        ax.grid(True, which='both', alpha=0.25); ax.legend(frameon=False)
        fig.tight_layout()
        plt.savefig(f'{stem}_uv_profiles.pdf', bbox_inches='tight', dpi=200)

    _plot_row1_2(out_stem)
    _plot_row3_4(out_stem)
    _plot_uv_profiles(out_stem)
    return prof


print('RT utilities loaded.')

RT utilities loaded.


In [ ]:

pairs = pick_random_pairs(FITS_ROOT, n=N_PAIRS, seed=RANDOM_SEED)
print(f'Processing {len(pairs)} pairs...')

blur_profiles = []
for i, (RAW, T50) in enumerate(pairs, 1):
    stem = f'explore_blurring_outputs/{Path(RAW).stem}'
    print(f'\n[{i}/{len(pairs)}] {Path(RAW).stem}')
    prof = make_blur_quicklook(RAW, t50_fits=T50, out_stem=stem,
                               fov_arcmin=Blur_FOV_ARCMIN)
    blur_profiles.append(prof)

# ---- Stacked UV summary (if >1 source) ----
if len(blur_profiles) > 1:
    r = blur_profiles[0]['r_kla']
    def _stack(key):
        Y = np.vstack([p[key] for p in blur_profiles])
        return np.nanmedian(Y, axis=0), np.nanstd(Y, axis=0)

    fig, ax = plt.subplots(figsize=(6.6, 4.6))
    for key, color, label in [
        ('med_T',  'C0', r'$|\mathcal{F}(T)|$'),
        ('med_RT', 'C1', r'$|\mathcal{F}(I)W|$'),
        ('med_IG', 'C2', r'$|\mathcal{F}(I*G)|$'),
    ]:
        mu, sd = _stack(key)
        ax.fill_between(r, mu-sd, mu+sd, alpha=0.18, color=color)
        ls = '--' if key=='med_T' else '-'
        ax.plot(r, mu, color=color, lw=1.8, ls=ls, label=label)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel(r'baseline $r$ (kλ)'); ax.set_ylabel('amplitude (Jy)')
    ax.set_title(f'Stacked UV profiles ({len(blur_profiles)} sources, median ± 1σ)')
    ax.grid(True, which='both', alpha=0.25); ax.legend(frameon=False)
    fig.tight_layout()
    plt.savefig('explore_blurring_outputs/uv_profiles_stack.pdf', bbox_inches='tight', dpi=200)

Processing 276 pairs (full dataset)...

[1/276] PSZ2G023.17+86.71


/tmp/ipykernel_27143/2505884656.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_27143/2505884656.py:206: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



[2/276] PSZ2G031.93+78.71


/tmp/ipykernel_27143/2505884656.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_27143/2505884656.py:206: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



[3/276] PSZ2G033.81+77.18


...

/tmp/ipykernel_27143/2505884656.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_27143/2505884656.py:206: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



[16/276] PSZ2G055.59+31.85


/tmp/ipykernel_27143/2505884656.py:178: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/tmp/ipykernel_27143/2505884656.py:206: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



[17/276] PSZ2G055.80+32.90


KeyboardInterrupt: 

## 7. Visual QC via Data Loader

Load images using `dcreclass.data.load_galaxies` and display side-by-side panels
for `VERSIONS[0]` (left) and `VERSIONS[1]` (right) for the first `N_SHOW` sources.

In [24]:
from dcreclass.data import load_galaxies

(train_images, train_labels,
 eval_images,  eval_labels,
 train_fns,    eval_fns) = load_galaxies(
    galaxy_classes=GALAXY_CLASSES,
    versions=VERSIONS,
    fold=5,
    NORMALISE=True,
    USE_GLOBAL_NORMALISATION=False,
    percentile_lo=30,
    percentile_hi=99,
    AUGMENT=False,
    PRINTFILENAMES=True,
    crop_mode=CROP_MODE,
    blur_method=BLUR_METHOD,
)

n_show = min(N_SHOW, len(train_images))
print(f'Loaded {len(train_images)} train / {len(eval_images)} eval images. Showing {n_show}.')

✓ Loading data from cache: ./.cache/images/cache_cls50_51_verBlur50kpc_T50kpc_f5_dd3322f4f0c421fc.pt
Loaded 144 train / 37 eval images. Showing 10.


In [25]:
import re as _re, os as _os

def _src_name(s):
    b = _os.path.splitext(_os.path.basename(str(s)))[0]
    return _re.sub(r'(?:T\d+kpc(?:SUB)?)$', '', b)

row_titles = [_src_name(train_fns[i]) for i in range(n_show)]
is_multi = isinstance(VERSIONS, (list, tuple)) and len(VERSIONS) > 1

if is_multi:
    ncols = len(VERSIONS)
    fig, axes = plt.subplots(n_show, ncols,
                             figsize=(3.0*ncols, 2.6*n_show),
                             constrained_layout=True)
    if n_show == 1: axes = [axes]
    for i in range(n_show):
        cube = train_images[i]  # shape: [T, 1, H, W]
        for j, ver in enumerate(VERSIONS):
            img = cube[j, 0].detach().cpu().numpy()
            axes[i][j].imshow(img, cmap='viridis', origin='lower')
            axes[i][j].set_title(f'{row_titles[i]} — {ver}', fontsize=8)
            axes[i][j].set_xticks([]); axes[i][j].set_yticks([])
else:
    fig, axes = plt.subplots(n_show, 1,
                             figsize=(3.0, 2.6*n_show),
                             constrained_layout=True)
    if n_show == 1: axes = [axes]
    for i in range(n_show):
        img = train_images[i][0].detach().cpu().numpy()
        axes[i].imshow(img, cmap='viridis', origin='lower')
        axes[i].set_title(f'{row_titles[i]} — {VERSIONS}', fontsize=8)
        axes[i].set_xticks([]); axes[i].set_yticks([])

plt.savefig('explore_blurring_outputs/samples_Blur_vs_T.pdf', bbox_inches='tight', dpi=200)

## 8. Multi-Scale Blurring Visual Comparison

Reads existing processed FITS files and displays a grid:
rows = sources, columns = RAW + T25 + Blur25 + T50 + Blur50 + T100 + Blur100.
No on-the-fly computation — requires processed files in `PROCESSED_DIR`.

In [ ]:
# ---- Section 8: multi-scale blurring grid ----

def _proc_path(slug, image_type):
    """Build path to a processed FITS file."""
    return Path(PROCESSED_DIR) / f"{slug}_{image_type}_fmt_128x128_{FILE_SUFFIX}.fits"


# --- 1. Discover available sources from RAW processed files ---
raw_files = sorted(Path(PROCESSED_DIR).glob(f'*_RAW_fmt_128x128_{FILE_SUFFIX}.fits'))
all_slugs = [f.name.split('_RAW_')[0] for f in raw_files]
print(f'Found {len(all_slugs)} sources with processed RAW files in:\n  {PROCESSED_DIR}')

if not all_slugs:
    print('No processed files found — check PROCESSED_DIR, CROP_MODE, BLUR_METHOD.')
else:
    import random as _random
    rng8 = _random.Random(RANDOM_SEED)
    sample_slugs = all_slugs if len(all_slugs) <= N_BLURRING else rng8.sample(all_slugs, N_BLURRING)

    # --- 2. Build column definitions ---
    col_defs = [('RAW', 'RAW')]  # (label, image_type)
    for s in SCALES:
        col_defs.append((f'T{s}kpc',         f'T{s}kpc'))
        col_defs.append((f'{BLUR_TAG}{s}kpc', f'{BLUR_TAG}{s}kpc'))
    n_cols = len(col_defs)

    # --- 3. Load images ---
    grid_data = []   # list of {label: arr or None}
    for slug in sample_slugs:
        row = {}
        for label, itype in col_defs:
            p = _proc_path(slug, itype)
            if p.exists():
                try:
                    arr, _ = load_fits(p)
                    row[label] = np.asarray(arr, float)
                except Exception as e:
                    print(f'[warn] {p.name}: {e}')
                    row[label] = None
            else:
                row[label] = None
        grid_data.append((slug, row))

    # --- 4. Display grid ---
    fig, axes = plt.subplots(N_BLURRING, n_cols,
                             figsize=(2.2 * n_cols, 2.4 * N_BLURRING),
                             constrained_layout=True)
    if N_BLURRING == 1:
        axes = [axes]

    for i, (slug, row) in enumerate(grid_data):
        for j, (label, _) in enumerate(col_defs):
            ax = axes[i][j]
            arr = row[label]
            if arr is not None and np.any(np.isfinite(arr)):
                vmin, vmax = robust_vmin_vmax(arr, lo=30, hi=99)
                ax.imshow(arr, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
            else:
                ax.set_facecolor('0.2')
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, color='white', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            if i == 0:
                ax.set_title(label, fontsize=9)
            if j == 0:
                ax.set_ylabel(slug[:20], fontsize=7, rotation=90, labelpad=2)

    fig.suptitle(f'Multi-scale blurring comparison  ({CROP_MODE} / {BLUR_METHOD})',
                 fontsize=11)
    plt.savefig(f'explore_blurring_outputs/sources_vs_versions.pdf', bbox_inches='tight', dpi=200)
    plt.close(fig)
    print(f'Grid saved to explore_blurring_outputs/source_vs_versions.pdf')

Found 276 sources with processed RAW files in:
  /users/mbredber/scratch/data/PSZ2/beam_crop/circular/fits_files
Grid saved to explore_blurring_outputs/source_vs_versions.pdf
